In [1]:
import gzip, re, os
import pandas as pd

# ---- 1. Transcript table, read from the headers of the primary-only FASTA ----
pattern = re.compile(
    r"^>(\S+) \S+ \S+:GRCh38:([^:]+):(\d+):(\d+):(-?1) "
    r"gene:(\S+) gene_biotype:(\S+) transcript_biotype:(\S+)"
    r"(?: gene_symbol:(\S+))?"
)
rows = []
with gzip.open("../data/ref/cdna.primary.fa.gz", "rt") as f:
    for line in f:
        if line.startswith(">"):
            m = pattern.match(line)
            if m:
                rows.append(m.groups())

tx = pd.DataFrame(rows, columns=[
    "transcript_id", "chrom", "start", "end", "strand",
    "gene_id", "gene_biotype", "transcript_biotype", "gene_symbol",
])
tx["start"] = tx["start"].astype(int)
tx["end"] = tx["end"].astype(int)
print("transcripts parsed:", len(tx), "(expect 187056)")

# ---- 2. Keep protein-coding genes on chr1-22, X, Y (MT dropped here) ----
keep = [str(i) for i in range(1, 23)] + ["X", "Y"]
tx_keep = tx[tx["chrom"].isin(keep) & (tx["gene_biotype"] == "protein_coding")]

genes = tx_keep.groupby("gene_id").agg(
    chrom=("chrom", "first"),
    start=("start", "min"),
    end=("end", "max"),
    symbol=("gene_symbol", "first"),
    n_chrom=("chrom", "nunique"),
)
assert (genes["n_chrom"] == 1).all()   # every gene on exactly one chromosome
print("genes kept:", len(genes))

# ---- 3. Sum transcript counts to gene counts for each sample ----
samples = {
    "CSES9": "SRR3501465",
    "CSES7": "SRR3501466",
    "CSES22": "SRR3501467",
    "CSES21": "SRR3501468",
    "HUES14": "SRR3501469",
}
counts = {}
for name, run in samples.items():
    q = pd.read_csv(f"../data/quant/primary/{run}/abundance.tsv", sep="\t")
    total = q["est_counts"].sum()
    q = q.merge(tx_keep[["transcript_id", "gene_id"]],
                left_on="target_id", right_on="transcript_id")
    counts[name] = q.groupby("gene_id")["est_counts"].sum()
    print(f"{name}: kept {q['est_counts'].sum() / total:.1%} of estimated counts")
counts = pd.DataFrame(counts)
print("count table shape:", counts.shape)

# ---- 4. Save ----
os.makedirs("../data/processed", exist_ok=True)
counts.to_csv("../data/processed/gene_counts.csv")
genes.to_csv("../data/processed/genes.csv")

# ---- 5. Marker check ----
cpm = counts / counts.sum() * 1e6
markers = ["POU5F1", "NANOG", "LIN28A", "SOX2", "DNMT3B",
           "SOX17", "PAX6", "GATA6", "TBXT"]
idx = genes.index[genes["symbol"].isin(markers)]
out = cpm.reindex(idx)
out.insert(0, "symbol", genes.loc[idx, "symbol"])
out.round(1).sort_values("symbol")

transcripts parsed: 187056 (expect 187056)
genes kept: 20052
CSES9: kept 96.0% of estimated counts
CSES7: kept 91.0% of estimated counts
CSES22: kept 97.9% of estimated counts
CSES21: kept 96.6% of estimated counts
HUES14: kept 95.8% of estimated counts
count table shape: (20052, 5)


,symbol,CSES9,CSES7,CSES22,CSES21,HUES14
gene_id,,,,,,
ENSG00000088305.19,DNMT3B,3067.4,1767.6,865.2,590.6,2848.8
ENSG00000141448.11,GATA6,0.3,44.2,6.1,12.8,0.9
ENSG00000131914.11,LIN28A,1861.7,1443.7,2096.8,1312.4,1414.1
ENSG00000111704.11,NANOG,135.2,108.8,21.1,73.2,150.0
ENSG00000007372.25,PAX6,0.2,0.9,0.2,2.1,0.3
ENSG00000204531.21,POU5F1,1517.7,1101.5,759.9,773.6,1897.7
ENSG00000164736.6,SOX17,0.0,32.1,2.5,6.5,0.4
ENSG00000181449.4,SOX2,210.0,210.9,194.3,142.4,244.5
ENSG00000164458.10,TBXT,0.2,6.3,0.9,7.1,0.3
